In [ ]:
#------------------------------------------------------------------------------------------------------#
#
# Code:      CC01_CCAR_B01_model_DEV_01.ipynb
#
# Objective: Step B01: Data cleaning, outlier capping, etc. Prepare final DEV data and OOT data.
#
#            Jingru Chen
#            2026-03-22
#
#----------------------------------------------------------------------------------------------------#

# Step 1: Upload libraries

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

run_date="2026-03-22"

start = datetime.now( ZoneInfo("America/New_York"))

print( start.strftime("%Y-%m-%d %H:%M:%S %Z"))     # 2026-03-17 17:34:58 EDT
print( start.strftime("%Y-%m-%d %I:%M:%S %p %Z"))  # 2026-03-17 05:34:58 PM EDT

2026-03-22 13:27:09 EDT
2026-03-22 01:27:09 PM EDT


In [ ]:
myout= "/content/sample_data"

In [ ]:
pwd

'/content'

In [ ]:
cd /content/sample_data/

/content/sample_data


In [ ]:
ls -ltr

total 84032
-rwxr-xr-x 1 root root      962 Jan  1  2000 README.md*
-rwxr-xr-x 1 root root     1697 Jan  1  2000 anscombe.json*
-rw-r--r-- 1 root root  1706430 Mar 17 17:58 california_housing_train.csv
-rw-r--r-- 1 root root   301141 Mar 17 17:58 california_housing_test.csv
-rw-r--r-- 1 root root 36523880 Mar 17 17:58 mnist_train_small.csv
-rw-r--r-- 1 root root 18289443 Mar 17 17:58 mnist_test.csv
-rw-r--r-- 1 root root 29210174 Mar 22 17:21 CCAR_Mortgage_data_for_model_DEV_20260319_01.csv


# Step 2: Split into DEV vs. Validation subfiles

In [ ]:
df_mortgage= pd.read_csv( myout + "/CCAR_Mortgage_data_for_model_DEV_20260319_01.csv" )
df_mortgage= df_mortgage.drop( columns= ['Unnamed: 0'] )

df_mortgage.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67957 entries, 0 to 67956
Data columns (total 44 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   loan_id                           67957 non-null  int64  
 1   origination_date                  67957 non-null  object 
 2   report_date                       67957 non-null  object 
 3   num_payments                      67957 non-null  int64  
 4   original_balance                  67957 non-null  float64
 5   current_balance_bk                67957 non-null  float64
 6   EAD                               67957 non-null  float64
 7   credit_score_orig                 67957 non-null  float64
 8   loan_to_value_orig                67957 non-null  float64
 9   interest_rate                     67957 non-null  float64
 10  product_type                      67957 non-null  object 
 11  ever_defaulted                    67957 non-null  bool   
 12  defa

In [ ]:
df_mortgage.columns

Index(['loan_id', 'origination_date', 'report_date', 'num_payments',
       'original_balance', 'current_balance_bk', 'EAD', 'credit_score_orig',
       'loan_to_value_orig', 'interest_rate', 'product_type', 'ever_defaulted',
       'default_date', 'PD', 'LGD', 'projected_loss', 'loan_term_months',
       'unemployment', 'gdp_growth_qoq', 'hpi_change', 'bbb_spread',
       'months_elapsed', 'current_balance', 'yrmo', 'report_yrmo',
       'default_yrmo', 'flag_default', 'flag_removal', 'delta_Unemployment1',
       'delta_Mortgage_rate1', 'delta_House_Price_Index__Level1',
       'delta_Unemployment3', 'delta_Mortgage_rate3',
       'delta_House_Price_Index__Level3', 'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12', 'delta_Mortgage_rate12',
       'delta_House_Price_Index__Level12', 'delta_Unemployment24',
       'delta_Mortgage_rate24', 'delta_House_Price_Index__Level24',
       'flag_merge'],
      dtype='object')

In [ ]:
x_list=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate', 'loan_term_months',
        'delta_Unemployment1',
       'delta_Mortgage_rate1', 'delta_House_Price_Index__Level1',
       'delta_Unemployment3', 'delta_Mortgage_rate3',
       'delta_House_Price_Index__Level3', 'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12', 'delta_Mortgage_rate12',
       'delta_House_Price_Index__Level12', 'delta_Unemployment24',
       'delta_Mortgage_rate24', 'delta_House_Price_Index__Level24']

x_list_v1=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate', 'loan_term_months',
        'delta_Unemployment1',
       'delta_Unemployment3', 'delta_Mortgage_rate3',
       'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12',
       'delta_Unemployment24' ]


y_list= ['flag_default']

In [ ]:
df_mortgage_model= df_mortgage.loc[df_mortgage.report_yrmo <= 201603]

df_mortgage_model.report_yrmo.value_counts()

,count
report_yrmo,
201401,2437
201402,2437
201403,2373
201404,2299
201405,2223
201406,2149
201407,2088
201408,2025
201409,1961


In [ ]:
pd.crosstab( index= df_mortgage_model['report_yrmo'], columns= df_mortgage_model['flag_default'],
            margins=True)

flag_default,0,1,All
report_yrmo,,,
201401,2437,0,2437
201402,2373,64,2437
201403,2299,74,2373
201404,2223,76,2299
201405,2149,74,2223
201406,2088,61,2149
201407,2025,63,2088
201408,1961,64,2025
201409,1900,61,1961


In [ ]:
df_mortgage_model.shape

(47119, 44)

In [ ]:
df_mortgage_model.loc[df_mortgage_model.loan_id== 5000]

,loan_id,origination_date,report_date,num_payments,original_balance,current_balance_bk,EAD,credit_score_orig,loan_to_value_orig,interest_rate,...,delta_Unemployment6,delta_Mortgage_rate6,delta_House_Price_Index__Level6,delta_Unemployment12,delta_Mortgage_rate12,delta_House_Price_Index__Level12,delta_Unemployment24,delta_Mortgage_rate24,delta_House_Price_Index__Level24,flag_merge
67908,5000,2012-08-31,2014-01-31,17,42239.85,19981.14,19981.14,698.0,85.751116,3.023,...,-0.061156,0.028032,0.040925,-0.120109,0.227760,0.102066,-0.197474,0.110798,0.195598,both
67909,5000,2012-08-31,2014-02-28,18,42239.85,19981.14,19981.14,698.0,85.751116,3.023,...,-0.067570,-0.008558,0.038174,-0.136627,0.233053,0.096899,-0.209510,0.129190,0.193900,both
67910,5000,2012-08-31,2014-03-31,19,42239.85,19981.14,19981.14,698.0,85.751116,3.023,...,-0.086987,-0.018509,0.034537,-0.160984,0.226092,0.089271,-0.235664,0.122672,0.189297,both
67911,5000,2012-08-31,2014-04-30,20,42239.85,19981.14,19981.14,698.0,85.751116,3.023,...,-0.111995,-0.016194,0.030270,-0.185762,0.196596,0.080151,-0.266972,0.102883,0.183267,both
67912,5000,2012-08-31,2014-05-31,21,42239.85,19981.14,19981.14,698.0,85.751116,3.023,...,-0.127696,-0.018289,0.025937,-0.197258,0.136054,0.071646,-0.286775,0.091942,0.177840,both
67913,5000,2012-08-31,2014-06-30,22,42239.85,19981.14,19981.14,698.0,85.751116,3.023,...,-0.124355,-0.035866,0.022192,-0.187868,0.050154,0.065107,-0.284965,0.105748,0.174346,both
67914,5000,2012-08-31,2014-07-31,23,42239.85,19981.14,19981.14,698.0,85.751116,3.023,...,-0.107862,-0.060020,0.019391,-0.169018,-0.031988,0.060316,-0.271899,0.135857,0.171995,both
67915,5000,2012-08-31,2014-08-31,24,42239.85,19981.14,19981.14,698.0,85.751116,3.023,...,-0.091343,-0.075823,0.018282,-0.158913,-0.084381,0.056456,-0.265191,0.164461,0.169246,both
67916,5000,2012-08-31,2014-09-30,25,42239.85,19981.14,19981.14,698.0,85.751116,3.023,...,-0.082612,-0.077021,0.018556,-0.169599,-0.095530,0.053093,-0.276623,0.175576,0.165221,both
67917,5000,2012-08-31,2014-10-31,26,42239.85,19981.14,19981.14,698.0,85.751116,3.023,...,-0.080418,-0.070475,0.019901,-0.192413,-0.086669,0.050171,-0.299056,0.167078,0.160395,both


In [ ]:
# X = df_mortgage_model[ x_list_v1 ]

x_list_v2=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate',
        'delta_Unemployment1',
       'delta_Unemployment3',
       'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12',
       'delta_Unemployment24' ]

X = df_mortgage_model[ x_list_v2 ]

y = df_mortgage_model['flag_default']

print( "---------------- Type of X is ---: ", type(X) )
print( "---------------- Type of y is ---: ", type(y) )

# Split: 80% Dev (Train), 20% Validation (Test)
X_dev, X_val, y_dev, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y  # CRITICAL: Ensures both files have the same % of defaults
)

---------------- Type of X is ---:  <class 'pandas.core.frame.DataFrame'>
---------------- Type of y is ---:  <class 'pandas.core.series.Series'>


# Step 3: Run various models on DEV data

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1. Define and Fit the Model
# l1_ratio=0.5 provides a 50/50 mix of Lasso and Ridge
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        penalty='elasticnet',
        solver='saga',
        l1_ratio=0.5,
        class_weight='balanced', # This automatically adjusts weights based on frequency
        max_iter=10000,
        random_state=42
    ))
])


pipeline.fit(X_dev, y_dev)

# 2. Extract and Display Coefficients
model_step = pipeline.named_steps['model']
coef_df = pd.DataFrame({
    'Feature': X_dev.columns,
    'Coefficient': model_step.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print("--- Model Coefficients ---")
print(coef_df)

--- Model Coefficients ---
                            Feature  Coefficient
1                 credit_score_orig     0.454710
3                     interest_rate     0.036936
5               delta_Unemployment3     0.034008
9              delta_Unemployment12     0.024497
0                  original_balance    -0.002303
4               delta_Unemployment1    -0.068078
8   delta_House_Price_Index__Level6    -0.073572
7              delta_Mortgage_rate6    -0.087262
6               delta_Unemployment6    -0.147342
10             delta_Unemployment24    -0.199124
2                loan_to_value_orig    -0.463493


In [ ]:

# 1. Prepare data for VIF (Must include a constant/intercept)
X_vif = add_constant(X_dev)

# 2. Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(len(X_vif.columns))]

# 3. Filter out the constant and sort
vif_results = vif_data[vif_data['feature'] != 'const'].sort_values(by='VIF', ascending=False)

print("\n--- VIF Results ---")
print(vif_results)


--- VIF Results ---
                            feature       VIF
6               delta_Unemployment3  5.916619
10             delta_Unemployment12  5.434354
8              delta_Mortgage_rate6  3.010194
7               delta_Unemployment6  2.982508
5               delta_Unemployment1  2.753254
9   delta_House_Price_Index__Level6  2.224007
11             delta_Unemployment24  2.197970
2                 credit_score_orig  1.012885
3                loan_to_value_orig  1.011678
4                     interest_rate  1.003293
1                  original_balance  1.001573


In [ ]:
# 1. Generate Predictions
# For metrics, we need the hard classes (0 or 1)
y_pred = pipeline.predict(X_val)

# 2. Calculate Individual Metrics
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

# 3. Print the Comprehensive Classification Report
print("--- Classification Report ---")
print(classification_report(y_val, y_pred))

# 4. Display the Confusion Matrix
print("--- Confusion Matrix ---")
print(confusion_matrix(y_val, y_pred))

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.98      0.62      0.76      9172
           1       0.04      0.60      0.08       252

    accuracy                           0.62      9424
   macro avg       0.51      0.61      0.42      9424
weighted avg       0.96      0.62      0.74      9424

--- Confusion Matrix ---
[[5716 3456]
 [ 100  152]]


In [ ]:
# 1. Generate Predictions
# Get probabilities instead of hard predictions
y_proba = pipeline.predict_proba(X_val)[:, 1]

# Set a custom threshold based on your portfolio's average default rate
custom_threshold = 0.6
y_pred_new = (y_proba >= custom_threshold).astype(int)


# 2. Calculate Individual Metrics
accuracy = accuracy_score( y_val, y_pred_new )
precision = precision_score( y_val, y_pred_new )
recall = recall_score(y_val, y_pred_new )
f1 = f1_score( y_val, y_pred_new )

# 3. Print the Comprehensive Classification Report
print("--- Classification Report ---")
print(classification_report(y_val, y_pred_new ))

# 4. Display the Confusion Matrix
print("--- Confusion Matrix ---")
print(confusion_matrix(y_val, y_pred_new ))

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.98      0.82      0.89      9172
           1       0.06      0.39      0.10       252

    accuracy                           0.81      9424
   macro avg       0.52      0.60      0.50      9424
weighted avg       0.96      0.81      0.87      9424

--- Confusion Matrix ---
[[7531 1641]
 [ 154   98]]


In [ ]:
# Assuming your pipeline is named 'pipeline'
model = pipeline.named_steps['model']
coefs = pd.DataFrame({
    'Feature': X_dev.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

# Note: Only scaled features allow for direct comparison of coefficient magnitude.

coefs

,Feature,Coefficient
1,credit_score_orig,0.454710
3,interest_rate,0.036936
5,delta_Unemployment3,0.034008
9,delta_Unemployment12,0.024497
0,original_balance,-0.002303
4,delta_Unemployment1,-0.068078
8,delta_House_Price_Index__Level6,-0.073572
7,delta_Mortgage_rate6,-0.087262
6,delta_Unemployment6,-0.147342
10,delta_Unemployment24,-0.199124


# Step 4: Save the new PD model into a pickle file

In [ ]:
# Save the entire object to a file
joblib.dump(pipeline, 'ccar_pd_model_{}.pkl'.format(run_date) )

['ccar_pd_model_2026-03-22.pkl']

In [ ]:
from datetime import datetime
end = datetime.now(ZoneInfo("America/New_York"))
duration = end - start

print(f"Started:  {start}")
print(f"Finished: {end}")
print(f"\nDuration: {duration}")                    # 0:00:02.351234
print(f"Duration: {duration.total_seconds():.3f} seconds")

Started:  2026-03-22 13:27:09.680230-04:00
Finished: 2026-03-22 13:27:15.161350-04:00

Duration: 0:00:05.481120
Duration: 5.481 seconds
